In [1]:
%pip install torch numpy scipy scikit-learn matplotlib

  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp313-c


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Plan
We should train/test before choosing the final 'best' model.
1. Import
2. Preprocessing: normalize data
3. Train/test with models. With parameter tuning with manual grid search. Find best number of past values to input.
4. Predict: 200 recursivily, scale back and output.
5. Get ready: report MAE and MSE, plot comparison.

### NN's all need time-series sequences: 
Long Short-Term Memory LSTM, a bit complex but sure good "LSTMs are designed to capture both short- and long-term dependencies in sequential data."
GRU is simpler and faster maybe worse
Use loss function just MSELoss probably
Optimizer idk. 

(python 3.13.7)

In [3]:
from scipy import io
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Load the MAT file
mat_data = io.loadmat('Xtrain.mat')

# Keys: dict_keys(['__header__', '__version__', '__globals__', 'Xtrain'])
# print(mat_data.keys())

data = mat_data['Xtrain'].flatten()
scaler = MinMaxScaler()                    # could chooose different scaler
data_scaled = scaler.fit_transform(data.reshape(-1,1)).flatten()

# Make time interval chunks for x and y
def create_intervals(data, interval_size):
    x, y = [],[]
    for i in range(len(data)-interval_size):
        x.append(data[i:i+interval_size])
        y.append(data[i+interval_size])
    return np.array(x), np.array(y)

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")       # torch cuda if possible (sure)

# LSTM
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, sequence_length):
        super(SimpleGRU, self).__init__()
        self.hidden_size  = hidden_size
        self.num_layers = num_layers
        
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc1 = nn.Linear(hidden_size * sequence_length, num_classes)
    
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        
        out,_ = self.gru(x, h0)
        out = out.reshape(out.shape[0], -1)
        out = self.fc1(out)
        return out

# Training.. epochs can be increased ofc
def train_model(x,y,learning_rate,nepoch, input_size, sequence_length, num_layers,hidden_size):
    x = torch.tensor(x, dtype=torch.float32).unsqueeze(-1).to(device)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1).to(device)

    model = SimpleGRU(input_size, hidden_size, num_layers, 1, sequence_length).to(device)
    lossfunction = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(),lr=learning_rate)

    for epoch in range(nepoch):
        model.train()
        optimizer.zero_grad()
        output = model(x)
        loss = lossfunction(output,y)
        loss.backward()
        optimizer.step()
    
    return model, loss.item()

In [ ]:
# Tuning on hidden_sizes, interval_size and learning_rate. Values? 
interval_sizes = [3,4,5,10,15,20,25,30,40]
hidden_sizes = [8,12,16,20,24,32,40,48]
learning_rates = [0.01,0.02,0.025,0.03,0.035,0.04,0.045,0.05]      # meer?

input_size=28
sequence_length =28
num_layers=2

best_loss = float("inf")
best_config = None
best_model= None

# Try configurations
for interval in interval_sizes:
    x,y = create_intervals(data_scaled,interval)
    for hsize in hidden_sizes:
        for lr in learning_rates:
            model, loss = train_model(x,y,lr, nepoch=20, input_size=input_size, sequence_length=sequence_length, num_layers=num_layers, hidden_size=hsize)
            print(f"interval={interval} hidden_size={hsize} learning_rate={lr} loss={loss}")
            # update best
            if (loss < best_loss):
                best_loss = loss
                best_config = (interval, hsize, lr)
                best_model= model

print(f"Best parameter configuration: {best_config}")
best_model.eval()

ValueError: input_size must be greater than zero

First run: interval_sizes = [5,10,15,20,25,30], hidden_sizes = [32,40,48,56,64], learning_rates = [0.001,0.00075,0.0005] we got best parameter configuration: (5, 32, 0.001)
Second run: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.002,0.0015,0.001,0.0005] we got best parameter configuration: (20, 24, 0.002)     - conclusion lr can be bigger
Third run: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.003,0.0025,0.002,0.0015,0.001] we got best parameter configuration: (30, 48, 0.003)           - still change. 
Foruth: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.005,0.004,0.003,0.0025,0.002] we got best parameter configuration: (5, 48, 0.005)            - zo afhankelijk wat
Fifth: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.005,0.01] we got best parameter configuration: (5, 16, 0.01)            -lr moet gewoon veel hoger zijn
Sixth: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [16,24,32,40,48] learning_rates = [0.005,0.01,0.05,0.1] best parameter configuration: (25, 16, 0.05)          - eindelijk lr range
Seventh: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [8,12,16,20,24,32,40,48] learning_rates = [0.01,0.025,0.05,0.075,0.1] best parameter configuration: (25, 16, 0.025)      - chill top dit is het
Laatste: interval_sizes = [3,4,5,10,15,20,25,30,40] hidden_sizes = [8,12,16,20,24,32,40,48] learning_rates = [0.01,0.02,0.025,0.03,0.035,0.04,0.045,0.05] best parameter configuration: (20, 32, 0.04) - zo veranderlijk

In [ ]:
# Prediction with 200 steps
input_sequence = data_scaled[-best_config[0]:]
preds = []

for _ in range(200):
    x = torch.tensor(input_sequence, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
    # reduce memory 
    with torch.no_grad():
        pred = model(x).item()
    preds.append(pred)
    # move to next
    input_sequence = np.append(input_sequence[1:],pred)

# scale back and output as csv
preds = scaler.inverse_transform(np.array(preds).reshape(-1,1)).flatten()
np.savetxt("200predictions.csv", preds, delimiter=",")

NameError: name 'best_config' is not defined

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# May 8th test
test_data = io.loadmat("Xtest.mat")
yreal = test_data["Xtest"].flatten()

mae = mean_absolute_error(yreal[:200],preds)
mse = mean_squared_error(yreal[:200],preds)
print(f"MAE: {mae}")
print(f"MSE: {mse}")

# plot
plt.figure()
plt.plot(yreal[:200], label="Real")
plt.plot(preds, label="Predicted")
plt.legend()
plt.title("Actual and predicted laser measurements")
plt.show

FileNotFoundError: [Errno 2] No such file or directory: 'Xtest.mat'